problem: https://judge.nitro-ai.org/competitions/nitro/pre-iaio-2026/4/view

In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from transformers.models.auto import AutoTokenizer, AutoModel
import seaborn as sns
import torch

train_df = pd.read_csv('train_data.csv')
emoji_queries = pd.read_csv('test_data.csv')
train_df.head(5)

,sentence_id,text
0,1000,"Alice looked all round the table, but there wa..."
1,1001,This time Alice waited patiently until it chos...
2,1002,Off with his head!’” “How dreadfully savage!”...
3,1003,"either the locks were too large, or the key wa..."
4,1004,"“I mean, what makes them so shiny?” Alice loo..."


In [86]:
examples = pd.read_csv('support_examples.csv')
col = []
for ex in examples.values:
    a = train_df[train_df['sentence_id'] == ex[1]]['text'].iloc[0]
    print(a)
    col.append(a)
examples['seq'] = col
examples

The cat sat on the mat.
I love drinking coffee in the morning.
The rocket launched into space.
Please call me on my cell phone.
It is raining cats and dogs.
Time is money.
The early bird catches the worm.


,emoji_sequence,sentence_id,seq
0,🐈🧘🧶,1607,The cat sat on the mat.
1,❤️☕🌅,1630,I love drinking coffee in the morning.
2,🚀🌌☄️,1150,The rocket launched into space.
3,🙏📞📱,1527,Please call me on my cell phone.
4,🌧️🐈🐕,1101,It is raining cats and dogs.
5,⏳💰💸,1454,Time is money.
6,🌅🐦🐛,1241,The early bird catches the worm.


In [ ]:
examples_text = ' '.join([f'emojis: {e[0]} sentence: {e[2]}' for e in examples.values])
examples_text

'emojis: 🐈🧘🧶 sentence: The cat sat on the mat. emojis: ❤️☕🌅 sentence: I love drinking coffee in the morning. emojis: 🚀🌌☄️ sentence: The rocket launched into space. emojis: 🙏📞📱 sentence: Please call me on my cell phone. emojis: 🌧️🐈🐕 sentence: It is raining cats and dogs. emojis: ⏳💰💸 sentence: Time is money. emojis: 🌅🐦🐛 sentence: The early bird catches the worm.'

In [89]:
model_path = 'Qwen/Qwen3-Embedding-0.6B'
device = 'cuda'
BATCH_SIZE = 16

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModel.from_pretrained(model_path).to(device)
model.eval()
def last_token_pool(last_hidden_states, attention_mask):
    left_padding = attention_mask[:, -1].sum() == attention_mask.shape[0]
    if left_padding:
        return last_hidden_states[:, -1]
    else:
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size = last_hidden_states.shape[0]
        return last_hidden_states[
            torch.arange(batch_size, device=last_hidden_states.device), sequence_lengths
        ]
def make_embs(list, is_query):
    all_embs = []
    if is_query:
        list = [f'Given a sequence of emojis, find an English phrase that fits it best, it could be a direct translation, an idiom or slang. Examples: {examples_text} Query: {item}' for item in list]
    with torch.no_grad():
        for i in range(0, len(list), BATCH_SIZE):
            inputs = tokenizer(list[i: i+BATCH_SIZE], padding=True, return_tensors='pt').to(device)
            # sentence_emb = model(**inputs).last_hidden_state[:, 0, :]
            sentence_emb = last_token_pool(model(**inputs).last_hidden_state, inputs['attention_mask'])
            sentence_emb = torch.nn.functional.normalize(sentence_emb, dim=1)
            all_embs.extend([*sentence_emb.cpu()])

    all_embs = torch.stack(all_embs)
    return all_embs


sen_embs = make_embs(train_df['text'].tolist(), False)
emoji_embs = make_embs(examples['emoji_sequence'].tolist(), True)
sen_embs.shape 

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

torch.Size([688, 1024])

In [90]:
sim_mat = emoji_embs @ sen_embs.T
preds = sim_mat.argmax(dim=1)
preds

tensor([425, 123, 150, 123, 425, 454, 241])

In [91]:
train_df.iloc[preds]

,sentence_id,text
425,1425,Let the cat out of the bag.
123,1123,"Run home this moment, and fetch me a pair of g..."
150,1150,The rocket launched into space.
123,1123,"Run home this moment, and fetch me a pair of g..."
425,1425,Let the cat out of the bag.
454,1454,Time is money.
241,1241,The early bird catches the worm.
